# Lesson 3, Exercise 3: Tuning Speculation with GPT-2 - Impact of Draft Length (`K`)

**Goal:**
The purpose of this exercise is to implement a simplified speculative decoding loop using readily available GPT-2 models and to investigate how a key hyperparameter – the draft length `K` (the number of tokens speculatively generated by the draft model) – influences the overall efficiency of the generation process. You will measure performance in terms of both wall-clock time and the number of computationally expensive forward passes through the target model.

## 2. Imports and Configuration

In [1]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
import time
import pandas as pd

TARGET_MODEL_NAME = "gpt2-medium"
DRAFT_MODEL_NAME = "gpt2" # Standard small GPT-2

PROMPT_TEXT = "Artificial intelligence is rapidly transforming our world by"
MAX_TOTAL_TOKENS_TO_GENERATE = 100 # Total new tokens to generate for each run
K_VALUES_TO_TEST = [1, 2, 3, 4, 5, 8]

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cpu


## 3. Load Models and Tokenizer

In [2]:
tokenizer = None
target_model = None
draft_model = None

# gpt2 and gpt2-medium share the exact same BPE vocabulary -> one tokenizer serves both.
tokenizer = AutoTokenizer.from_pretrained(TARGET_MODEL_NAME)
if tokenizer.pad_token is None: tokenizer.pad_token = tokenizer.eos_token

# Target model (gpt2-medium, 355M): the model whose distribution we want to reproduce exactly
target_model = AutoModelForCausalLM.from_pretrained(TARGET_MODEL_NAME).to(device)
target_model.eval()

# Draft model (gpt2, 124M): cheap proposer
draft_model = AutoModelForCausalLM.from_pretrained(DRAFT_MODEL_NAME).to(device)
draft_model.eval()

if not all([tokenizer, target_model, draft_model]):
    raise ValueError("One or more models/tokenizer failed to load. Check TODOs.")
print("Models and tokenizer loaded successfully.")

initial_prompt_ids = tokenizer.encode(PROMPT_TEXT, return_tensors="pt").to(device)


Loading weights:   0%|          | 0/292 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Models and tokenizer loaded successfully.


## 4. Baseline Autoregressive Generation (Using Target Model)

In [3]:
print("\n--- Running Baseline Autoregressive Generation ---")
baseline_generated_ids = initial_prompt_ids.clone()
baseline_target_passes = 0
start_time_baseline = time.time()

with torch.no_grad():
    for _ in range(MAX_TOTAL_TOKENS_TO_GENERATE):
        # 1. Run the target model on the full current sequence (no KV cache -> one full pass per token,
        #    which keeps the "forward pass" accounting identical to the speculative loop below)
        outputs = target_model(baseline_generated_ids)
        # 2. Logits for the next position
        next_token_logits = outputs.logits[:, -1, :]
        # 3. Greedy choice
        next_token_id = torch.argmax(next_token_logits, dim=-1, keepdim=True)  # shape (1, 1)
        # 4. Append
        baseline_generated_ids = torch.cat([baseline_generated_ids, next_token_id], dim=1)
        # 5. Count the target pass
        baseline_target_passes += 1
        if next_token_id.item() == tokenizer.eos_token_id:
            break
end_time_baseline = time.time()
baseline_time = end_time_baseline - start_time_baseline
baseline_output_text = tokenizer.decode(baseline_generated_ids[0], skip_special_tokens=True)

print(f"Baseline Time: {baseline_time:.4f} s")
print(f"Baseline Target Model Passes: {baseline_target_passes}")
print(f"Baseline Output: {baseline_output_text}")



--- Running Baseline Autoregressive Generation ---


Baseline Time: 39.8799 s
Baseline Target Model Passes: 100
Baseline Output: Artificial intelligence is rapidly transforming our world by making it easier to create, share, and discover.

The future of AI is here.

The future of AI is here.

The future of AI is here.

The future of AI is here.

The future of AI is here.

The future of AI is here.

The future of AI is here.

The future of AI is here.

The future of AI is here.

The future of AI is here


## 5. Speculative Decoding Experiment

In [4]:
all_experiment_results = []

def target_single_step(context_ids):
    """Fallback: one greedy autoregressive step with the target model."""
    logits = target_model(context_ids).logits[:, -1, :]
    return torch.argmax(logits, dim=-1, keepdim=True)

if target_model and draft_model and tokenizer and initial_prompt_ids is not None:
    print("\n--- Running Speculative Decoding Experiments for different K values ---")

    for K_val in K_VALUES_TO_TEST:
        print(f"\nStarting Speculative Decoding with K = {K_val}")
        current_context_ids_spec = initial_prompt_ids.clone()
        num_total_new_tokens_generated = 0
        count_target_model_passes_spec = 0
        count_total_draft_tokens_verified_correctly = 0
        count_verification_steps_spec = 0 # How many times the target model was called to verify
        count_draft_passes = 0            # forward passes of the draft model

        # Timing Start
        if device.type == 'cuda': torch.cuda.synchronize()
        time_start_speculative = time.perf_counter()

        with torch.no_grad():
            while num_total_new_tokens_generated < MAX_TOTAL_TOKENS_TO_GENERATE:
                if current_context_ids_spec.shape[1] >= target_model.config.max_position_embeddings - K_val -1: # Ensure space for K draft + 1 target
                    print(f"K={K_val}: Approaching model's maximum length. Stopping early to prevent overflow.")
                    break

                # --- Draft Phase: the small model proposes up to K tokens autoregressively ---
                draft_tokens_generated_ids = torch.empty((1,0), dtype=torch.long, device=device)
                temp_context_for_draft = current_context_ids_spec.clone()
                for _ in range(K_val):
                    if temp_context_for_draft.shape[1] >= draft_model.config.max_position_embeddings:
                        break # Draft model also has a max length
                    draft_logits = draft_model(temp_context_for_draft).logits[:, -1, :]
                    count_draft_passes += 1
                    draft_next = torch.argmax(draft_logits, dim=-1, keepdim=True)
                    draft_tokens_generated_ids = torch.cat([draft_tokens_generated_ids, draft_next], dim=1)
                    temp_context_for_draft = torch.cat([temp_context_for_draft, draft_next], dim=1)
                    if draft_next.item() == tokenizer.eos_token_id:
                        break

                num_tokens_actually_drafted = draft_tokens_generated_ids.shape[1]
                if num_tokens_actually_drafted == 0:
                    # Nothing drafted: fall back to a single target step so we always make progress
                    nxt = target_single_step(current_context_ids_spec)
                    count_target_model_passes_spec += 1
                    current_context_ids_spec = torch.cat([current_context_ids_spec, nxt], dim=1)
                    num_total_new_tokens_generated += 1
                    if nxt.item() == tokenizer.eos_token_id:
                        break
                    continue

                # --- Verification Phase: ONE target pass over context + all K draft tokens ---
                verification_input_ids = torch.cat([current_context_ids_spec, draft_tokens_generated_ids], dim=1)

                target_verification_outputs = target_model(verification_input_ids)
                count_target_model_passes_spec += 1
                count_verification_steps_spec += 1
                target_logits = target_verification_outputs.logits  # (1, ctx+K, vocab)

                # The logits at position t predict token t+1. The draft token j (0-based) sits at absolute
                # index ctx_len + j, so the target's opinion about it is argmax(logits[ctx_len + j - 1]).
                ctx_len = current_context_ids_spec.shape[1]
                target_preferred_tokens_at_draft_positions = torch.argmax(
                    target_logits[0, ctx_len - 1 : ctx_len - 1 + num_tokens_actually_drafted, :], dim=-1
                ).tolist()

                # --- Acceptance Logic: accept the longest matching prefix (greedy speculative decoding) ---
                num_matched_tokens = 0
                draft_list = draft_tokens_generated_ids[0].tolist()
                for i in range(num_tokens_actually_drafted):
                    if draft_list[i] == target_preferred_tokens_at_draft_positions[i]:
                        num_matched_tokens += 1
                    else:
                        break
                count_total_draft_tokens_verified_correctly += num_matched_tokens

                accepted_tokens_for_this_step = draft_tokens_generated_ids[0, :num_matched_tokens]

                # Determine the next token: either the one from target model at mismatch, or one beyond matched sequence.
                if num_matched_tokens < num_tokens_actually_drafted:
                    # Mismatch: the target's own choice at the mismatch position is guaranteed correct -> use it
                    next_token_id_after_match = torch.tensor([[target_preferred_tokens_at_draft_positions[num_matched_tokens]]], device=device)
                else:
                    # All drafts accepted: bonus (K+1)-th token from the last logit row, for free
                    verification_idx_for_bonus_token = ctx_len + num_tokens_actually_drafted - 1
                    if verification_idx_for_bonus_token < target_logits.shape[1]:
                        next_token_id_after_match = torch.argmax(target_logits[0, verification_idx_for_bonus_token, :]).unsqueeze(0).unsqueeze(0)
                    else:
                        next_token_id_after_match = None

                if next_token_id_after_match is not None:
                     accepted_tokens_for_this_step = torch.cat([accepted_tokens_for_this_step, next_token_id_after_match.squeeze(0)])

                if accepted_tokens_for_this_step.numel() == 0:
                    # Cannot happen with the logic above (we always add at least the target's token), but keep a
                    # progress guarantee anyway
                    nxt = target_single_step(current_context_ids_spec)
                    count_target_model_passes_spec += 1
                    accepted_tokens_for_this_step = nxt.squeeze(0)

                # Do not overshoot the requested number of tokens (keeps runs comparable)
                remaining = MAX_TOTAL_TOKENS_TO_GENERATE - num_total_new_tokens_generated
                accepted_tokens_for_this_step = accepted_tokens_for_this_step[:remaining]

                current_context_ids_spec = torch.cat([current_context_ids_spec, accepted_tokens_for_this_step.unsqueeze(0)], dim=1)
                num_total_new_tokens_generated += accepted_tokens_for_this_step.shape[0]

                if tokenizer.eos_token_id in accepted_tokens_for_this_step:
                    print(f"K={K_val}: EOS token generated.")
                    break

        # Timing End
        if device.type == 'cuda': torch.cuda.synchronize()
        time_end_speculative = time.perf_counter()

        duration_speculative = time_end_speculative - time_start_speculative
        text_output_speculative = tokenizer.decode(current_context_ids_spec[0], skip_special_tokens=True)

        # Average tokens accepted per target verification pass (draft matches + 1 corrective/bonus token)
        avg_accepted_this_K = num_total_new_tokens_generated / count_target_model_passes_spec if count_target_model_passes_spec else float('nan')
        acceptance_rate = count_total_draft_tokens_verified_correctly / max(1, count_draft_passes)

        all_experiment_results.append({
            "K": K_val,
            "Time (s)": round(duration_speculative, 4),
            "Speed-up vs Baseline": round(baseline_time / duration_speculative, 2),
            "Target Passes": count_target_model_passes_spec,
            "Draft Passes": count_draft_passes,
            "Avg Accepted Tokens per Verification": round(avg_accepted_this_K, 2),
            "Draft Acceptance Rate": round(acceptance_rate, 2),
            "Identical to Baseline": text_output_speculative == baseline_output_text,
            "Output Text Sample": text_output_speculative[:150] + "..."
        })
        print(f"K={K_val}: Time={duration_speculative:.4f}s, Target Passes={count_target_model_passes_spec}, "
              f"Avg Accepted={avg_accepted_this_K:.2f}, acceptance rate={acceptance_rate:.2f}, "
              f"identical to baseline={text_output_speculative == baseline_output_text}")

        if torch.cuda.is_available():
            torch.cuda.empty_cache()
else:
    print("Skipping speculative decoding experiment as models/tokenizer were not properly loaded.")



--- Running Speculative Decoding Experiments for different K values ---

Starting Speculative Decoding with K = 1


K=1: Time=30.1769s, Target Passes=53, Avg Accepted=1.89, acceptance rate=0.91, identical to baseline=True

Starting Speculative Decoding with K = 2


K=2: Time=26.3327s, Target Passes=38, Avg Accepted=2.63, acceptance rate=0.82, identical to baseline=True

Starting Speculative Decoding with K = 3


K=3: Time=24.9639s, Target Passes=30, Avg Accepted=3.33, acceptance rate=0.81, identical to baseline=True

Starting Speculative Decoding with K = 4


K=4: Time=24.7350s, Target Passes=26, Avg Accepted=3.85, acceptance rate=0.75, identical to baseline=True

Starting Speculative Decoding with K = 5


K=5: Time=23.7738s, Target Passes=23, Avg Accepted=4.35, acceptance rate=0.70, identical to baseline=True

Starting Speculative Decoding with K = 8


K=8: Time=22.6807s, Target Passes=18, Avg Accepted=5.56, acceptance rate=0.57, identical to baseline=True


## 6. Display Final Results Summary

In [5]:
speculative_decoding_results_list = all_experiment_results
df_spec_results = pd.DataFrame(speculative_decoding_results_list)
pd.set_option("display.max_colwidth", 60)
print("\n\n--- Speculative Decoding Experiment Results Summary ---")
print(f"Baseline (autoregressive target only): {baseline_time:.4f} s, {baseline_target_passes} target passes")
print(df_spec_results.drop(columns=["Output Text Sample"]).to_string(index=False))




--- Speculative Decoding Experiment Results Summary ---
Baseline (autoregressive target only): 39.8799 s, 100 target passes
 K  Time (s)  Speed-up vs Baseline  Target Passes  Draft Passes  Avg Accepted Tokens per Verification  Draft Acceptance Rate  Identical to Baseline
 1   30.1769                  1.32             53            53                                  1.89                   0.91                   True
 2   26.3327                  1.51             38            76                                  2.63                   0.82                   True
 3   24.9639                  1.60             30            90                                  3.33                   0.81                   True
 4   24.7350                  1.61             26           104                                  3.85                   0.75                   True
 5   23.7738                  1.68             23           115                                  4.35                   0.70          

## 7. Analysis and Discussion

*Run on CPU (Intel i7-10610U, 4 cores), gpt2‑medium target / gpt2 draft, greedy decoding, 100 new tokens, no KV cache in either loop so that one "pass" = one full forward over the sequence.*

| K | Time (s) | Speed‑up | Target passes | Draft passes | Avg accepted / verification | Draft acceptance rate | Output == baseline |
|---|---|---|---|---|---|---|---|
| baseline | 39.88 | 1.00× | 100 | 0 | – | – | – |
| 1 | 30.18 | 1.32× | 53 | 53 | 1.89 | 0.91 | ✅ |
| 2 | 26.33 | 1.51× | 38 | 76 | 2.63 | 0.82 | ✅ |
| 3 | 24.96 | 1.60× | 30 | 90 | 3.33 | 0.81 | ✅ |
| 4 | 24.74 | 1.61× | 26 | 104 | 3.85 | 0.75 | ✅ |
| 5 | 23.77 | 1.68× | 23 | 115 | 4.35 | 0.70 | ✅ |
| 8 | 22.68 | 1.76× | 18 | 144 | 5.56 | 0.57 | ✅ |

1.  **Impact of `K` on Target Model Passes:**
    *   The baseline needs one target pass per token (100). With speculation each verification pass yields `matched + 1` tokens, so target passes fall monotonically with `K`: 53 (K=1) → 38 → 30 → 26 → 23 → **18 (K=8)**, a 5.6× reduction in expensive target forward passes.

2.  **Impact of `K` on Wall-Clock Time:**
    *   Wall‑clock time also improves monotonically here — 39.9 s → 30.2 s (K=1) → 22.7 s (K=8), a 1.76× speed‑up — but with strongly diminishing returns: K=1→2 saves 3.8 s, K=5→8 saves only 1.1 s. Each extra draft token costs a full gpt2 forward pass (draft passes rise from 53 to 144) while the probability that it is accepted keeps falling, so at some larger K the draft overhead exceeds the savings and time goes back up. For this pair on this CPU the optimum is at or slightly beyond K=8; on a GPU, where the target's cost is nearly independent of the number of verified tokens but the draft's sequential steps are latency‑bound, the sweet spot is usually K≈4–6.

3.  **Impact of `K` on Average Accepted Tokens:**
    *   Tokens accepted per verification rise from 1.89 (K=1) to 5.56 (K=8) — the *absolute* yield per target pass grows — while the acceptance *rate* per drafted token falls from 91 % to 57 %. Both facts follow from the acceptance being a prefix match: the chance that all of the first *j* draft tokens are right decays roughly geometrically (~0.85^j here), so long drafts add tokens with rapidly shrinking marginal value. A high average accepted count with a still‑healthy acceptance rate (K=3–5) is the signature of an efficient draft/target pairing.

4.  **Trade-offs of `K`:**
    *   Small K: few wasted draft passes, high acceptance rate, but many verification passes (little speed‑up). Large K: fewer target passes and more tokens per pass, but more draft compute, more rejected work, and (with a KV cache) more cache roll‑back. The right K depends on how well the draft mimics the target (acceptance rate α) and on the cost ratio c = draft/target per pass; the expected tokens per target pass is (1−α^{K+1})/(1−α), and one picks the K maximizing that divided by (1 + cK).

5.  **Comparison to Baseline:**
    *   The best configuration (K=8) generated the same 100 tokens with **18 instead of 100 target passes (−82 %)** and in **22.7 s instead of 39.9 s (1.76× faster)**. Crucially every K produced output *identical* to the baseline — speculative decoding with greedy acceptance is lossless with respect to the target model, so this speed‑up comes at zero quality cost.
